In [ ]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory ready at: {PROJECT_DIR}")

Mounted at /content/drive
Working directory ready at: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


In [ ]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory ready at: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory ready at: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


## Hugging Face Hub Token Configuration

To avoid unauthenticated request warnings and potentially speed up model and dataset downloads, it is recommended to configure a Hugging Face Hub access token. Follow the steps below:

1.  **Get your Access Token:**
    *   Go to [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
    *   Create a new token. A token with `read` permission is recommended for most download operations.

2.  **Add the Token to Colab Secrets:**
    *   In the left panel of Google Colab, click on the key icon (🔑) to open the "Secrets" interface.
    *   Click on "Add new secret".
    *   In the "Name" field, type `HF_TOKEN`.
    *   In the "Value" field, paste the token you got from Hugging Face.
    *   Make sure to enable the "Notebook access" toggle so the notebook can use this secret.

3.  **Run the Python cell below:**
    *   This cell will load the token from Colab secrets and set it as an environment variable, which will be used by Hugging Face libraries.

In [ ]:
# Import the necessary libraries
from google.colab import userdata
import os
from huggingface_hub import login # Imports the login function from Hugging Face Hub

# Load the Hugging Face token from Colab secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        # Try to login with the Hugging Face token
        login(token=hf_token, add_to_git_credential=False) # 'add_to_git_credential=False' to avoid asking for git credentials
        print("Hugging Face token successfully loaded and configured via huggingface_hub.login().")
    else:
        print("WARNING: The secret 'HF_TOKEN' was found, but it is empty or None. Please check the value in Colab secrets.")
        # If the token is empty/None, still set the environment variable as an empty string to avoid later errors
        os.environ['HF_TOKEN'] = ''
except userdata.SecretNotFoundError:
    print("WARNING: The secret 'HF_TOKEN' was not found. Please add your Hugging Face token to Colab secrets.")
    os.environ['HF_TOKEN'] = '' # Ensure the environment variable is set, even if empty
except Exception as e:
    print(f"An error occurred while loading or configuring the Hugging Face token: {e}")
    os.environ['HF_TOKEN'] = '' # Ensure the environment variable is set, even if empty

Hugging Face token successfully loaded and configured via huggingface_hub.login().


In [ ]:
from datasets import load_dataset, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
tokenizer_dir = os.path.join(PROJECT_DIR, "tokenizador")

# Checks if the tokenizer already exists and loads it, otherwise trains a new one.
if os.path.exists(tokenizer_dir) and os.path.isdir(tokenizer_dir) and \
   os.path.exists(os.path.join(tokenizer_dir, "vocab.json")) and \
   os.path.exists(os.path.join(tokenizer_dir, "merges.txt")):
    print(f"Tokenizer found at: {tokenizer_dir}. Loading existing tokenizer...")
    tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, local_files_only=True)
    print("Tokenizer successfully loaded!")
else:
    print(f"Tokenizer not found or incomplete at {tokenizer_dir}. Training a new tokenizer...")

    # 1. Load exact fractions directly to the local Colab cache (WITHOUT streaming=True)
    # Divided proportionally to sum up to 50,000 articles in total (50% EN, 25% PT, 25% ES)
    print("Downloading Wikipedia slices to local memory (This takes about 1-2 minutes)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:25000]")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:12500]")
    wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:12500]")

    # Merges the locally downloaded datasets and shuffles in RAM
    print("Mixing and preparing the data...")
    mixed_dataset = concatenate_datasets([wiki_en, wiki_pt, wiki_es])
    mixed_dataset = mixed_dataset.shuffle(seed=42)

    # Generator used to feed the tokenizer trainer directly from RAM
    def extract_text():
        for item in mixed_dataset:
            yield item["text"]

    print("Training the tokenizer... This should now take 2 to 3 minutes.")
    tokenizer_raw = ByteLevelBPETokenizer()
    tokenizer_raw.train_from_iterator(
        extract_text(),
        vocab_size=50257, # Classic GPT-2 default
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Saves the tokenizer to Drive
    # REMOVES the existing directory before creating again to force synchronization
    if os.path.exists(tokenizer_dir) and os.path.isdir(tokenizer_dir):
        print(f"Removing existing tokenizer directory: {tokenizer_dir}")
        shutil.rmtree(tokenizer_dir)

    os.makedirs(tokenizer_dir, exist_ok=True) # Creates the directory again
    tokenizer_raw.save_model(tokenizer_dir)

    # Converts to the format usable by Hugging Face Trainer
    tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")
    tokenizer.save_pretrained(tokenizer_dir)
    print(f"Tokenizer successfully saved at: {tokenizer_dir}")

Tokenizer found at: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/tokenizador. Loading existing tokenizer...
Tokenizer successfully loaded!


In [ ]:
import os

# Set BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset, interleave_datasets
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from transformers.trainer_utils import get_last_checkpoint


# ============================================================
# MAIA LITE PRETRAINING
# Conservative configuration for Google Colab L4 / A100 / V100
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
TOKENIZER_DIR = os.path.join(PROJECT_DIR, "tokenizador")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "checkpoints_pretreino")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "modelo_355M_final")

MAX_LENGTH = 1024
MAX_STEPS = 300_000


# ------------------------------------------------------------
# 1. GPU detection
# ------------------------------------------------------------

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    device = torch.device("cuda")

    props = torch.cuda.get_device_properties(0)
    gpu_memory_gb = props.total_memory / (1024 ** 3)

    print(f"GPU available: {gpu_name}")
    print(f"GPU memory: {gpu_memory_gb:.1f} GB")
else:
    gpu_name = "CPU"
    device = torch.device("cpu")

    print(
        "WARNING: No CUDA GPU available. "
        "Training will be extremely slow."
    )


# ------------------------------------------------------------
# 2. Tokenizer
# ------------------------------------------------------------

tokenizer = GPT2TokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True,
)

# GPT-style causal LM normally uses EOS as padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocabulary: {tokenizer.vocab_size:,}")
print(f"EOS token: {tokenizer.eos_token_id}")
print(f"PAD token: {tokenizer.pad_token_id}")


# ------------------------------------------------------------
# 3. Find latest checkpoint
# ------------------------------------------------------------

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"\nCheckpoint found: {last_checkpoint}")

    model = GPT2LMHeadModel.from_pretrained(
        last_checkpoint,
        local_files_only=True,
    )

else:
    print("\nNo checkpoint found. Creating Maia Lite from scratch.")

    config = GPT2Config(
        vocab_size=tokenizer.vocab_size,
        n_positions=MAX_LENGTH,
        n_ctx=MAX_LENGTH,
        n_embd=1024,
        n_layer=24,
        n_head=16,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False,
        tie_word_embeddings=True,
    )

    model = GPT2LMHeadModel(config)


# Important when using gradient checkpointing
model.config.use_cache = False

model.to(device)

print(f"Model parameters: {model.num_parameters():,}")


# ------------------------------------------------------------
# 4. Verify tied embeddings
# ------------------------------------------------------------

try:
    tied = (
        model.transformer.wte.weight.data_ptr()
        == model.lm_head.weight.data_ptr()
    )

    print(f"Input/output embeddings tied: {tied}")

except Exception as exc:
    print(f"Could not verify tied embeddings: {exc}")


# ------------------------------------------------------------
# 5. Gradient checkpointing
#
# KEEP ENABLED.
# Maia Lite previously experienced CUDA OOM on Colab.
# Reliability is more important than a small speed gain.
# ------------------------------------------------------------

model.gradient_checkpointing_enable()

print("Gradient checkpointing: ENABLED")


# ------------------------------------------------------------
# 6. Wikipedia streaming datasets
# ------------------------------------------------------------

print("\nLoading Wikipedia streams...")

wiki_en = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
)

wiki_pt = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
)

wiki_es = load_dataset(
    "wikimedia/wikipedia",
    "20231101.es",
    split="train",
    streaming=True,
)

dataset_mixed = interleave_datasets(
    [wiki_en, wiki_pt, wiki_es],
    probabilities=[0.50, 0.25, 0.25],
    seed=42,
)


# ------------------------------------------------------------
# 7. Tokenization
#
# IMPORTANT:
# We deliberately keep the current tokenization strategy while
# resuming this pretraining run.
#
# Changing to document packing halfway through the experiment
# changes the training-data distribution.
#
# Packing should be implemented and benchmarked separately for
# the next Maia pretraining run.
# ------------------------------------------------------------

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )


tokenized_dataset = dataset_mixed.map(
    tokenize_function,
    batched=True,
    remove_columns=["id", "url", "title", "text"],
)


# ------------------------------------------------------------
# 8. Causal language-model collator
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ------------------------------------------------------------
# 9. Conservative GPU profile
# ------------------------------------------------------------

batch_size = 1
gradient_accumulation = 16
use_bf16 = False
use_fp16 = True


if "A100" in gpu_name:

    # Still conservative because the same notebook may receive
    # different A100 memory configurations.
    batch_size = 8
    gradient_accumulation = 4

    use_bf16 = True
    use_fp16 = False


elif "L4" in gpu_name:

    # Known-safe configuration from the current Maia run.
    batch_size = 4
    gradient_accumulation = 8

    use_bf16 = True
    use_fp16 = False


elif "V100" in gpu_name:

    batch_size = 4
    gradient_accumulation = 8

    use_bf16 = False
    use_fp16 = True


elif "T4" in gpu_name:

    batch_size = 1
    gradient_accumulation = 16

    use_bf16 = False
    use_fp16 = True


effective_batch = batch_size * gradient_accumulation

print("\nTraining configuration")
print("----------------------")
print(f"GPU:                   {gpu_name}")
print(f"Micro batch:           {batch_size}")
print(f"Gradient accumulation: {gradient_accumulation}")
print(f"Effective batch:       {effective_batch}")
print(f"Context length:        {MAX_LENGTH}")
print(f"BF16:                  {use_bf16}")
print(f"FP16:                  {use_fp16}")


# ------------------------------------------------------------
# 10. Training arguments
# ------------------------------------------------------------

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    # Total training target
    max_steps=MAX_STEPS,

    # Memory-safe batching
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation,

    # Precision
    bf16=use_bf16,
    fp16=use_fp16,

    # Optimization
    learning_rate=4e-4,
    weight_decay=0.01,

    # Explicit scheduler
    lr_scheduler_type="linear",
    warmup_steps=3000,

    # Logging
    logging_steps=100,

    # Checkpoints
    save_steps=2000,
    save_total_limit=2,

    # Streaming dataset
    dataloader_num_workers=2,

    # IMPORTANT:
    # Keep this behavior because restarting a huge streaming
    # dataset and skipping tens of thousands of batches can take
    # a very long time in Colab.
    ignore_data_skip=True,

    # Avoid unnecessary integrations
    report_to="none",

    # Reduces CPU -> GPU transfer overhead
    dataloader_pin_memory=True,

    # Let Trainer remove unused dataset columns
    remove_unused_columns=True,
)


# ------------------------------------------------------------
# 11. Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


# ------------------------------------------------------------
# 12. CUDA diagnostics before training
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\nCUDA memory before training")
    print("---------------------------")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved:  {reserved:.2f} GB")


# ------------------------------------------------------------
# 13. Resume training
# ------------------------------------------------------------

print("\nStarting Maia Lite pretraining...")

if last_checkpoint is not None:

    print(f"Resuming Trainer state from:")
    print(last_checkpoint)

    trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

else:

    trainer.train()


# ------------------------------------------------------------
# 14. Save final consolidated model
# ------------------------------------------------------------

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("\n========================================")
print("MAIA LITE PRETRAINING COMPLETED")
print("========================================")
print(f"Final model saved to: {FINAL_MODEL_DIR}")

GPU available: NVIDIA L4
GPU memory: 22.0 GB
Tokenizer vocabulary: 50,257
EOS token: 2
PAD token: 1

Checkpoint found: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-106000


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Model parameters: 354,823,168
Input/output embeddings tied: True
Gradient checkpointing: ENABLED

Loading Wikipedia streams...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

...

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
106100,2.742261
106200,2.776634
106300,2.743527
106400,2.615269
106500,2.533116
106600,2.557037
106700,2.583932
106800,2.521002
106900,2.604137
107000,2.643016


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
106100,2.742261
106200,2.776634
106300,2.743527
106400,2.615269
106500,2.533116
106600,2.557037
106700,2.583932
106800,2.521002
106900,2.604137
107000,2.643016


KeyboardInterrupt: 

In [ ]:
import os
import torch
import numpy as np

from datasets import load_dataset, interleave_datasets
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
)
from torch.utils.data import DataLoader


# ============================================================
# MAIA LITE - STREAM LOSS DIAGNOSTIC
#
# Loads checkpoint-106000 and evaluates the Wikipedia stream
# WITHOUT training or modifying model weights.
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"

TOKENIZER_DIR = os.path.join(
    PROJECT_DIR,
    "tokenizador"
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints_pretreino",
    "checkpoint-106000"
)

MAX_LENGTH = 1024

MICRO_BATCH = 4

# Equivalent to Trainer gradient accumulation
GRAD_ACCUM = 8

# Trainer logged every 100 optimizer steps.
LOGGING_STEPS = 100

# Number of equivalent training steps to inspect.
# 3000 steps covers substantially beyond the observed transition.
TEST_STEPS = 3000


# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA GPU required."

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Tokenizer
# ------------------------------------------------------------

tokenizer = GPT2TokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ------------------------------------------------------------
# 3. Load checkpoint
# ------------------------------------------------------------

print("\nLoading checkpoint:")
print(CHECKPOINT_DIR)

model = GPT2LMHeadModel.from_pretrained(
    CHECKPOINT_DIR,
    local_files_only=True,
)

model.config.use_cache = False

model.to(device)
model.eval()

print(
    f"Parameters: {model.num_parameters():,}"
)


# ------------------------------------------------------------
# 4. Recreate EXACT Wikipedia stream
# ------------------------------------------------------------

print("\nLoading Wikipedia streams...")

wiki_en = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
)

wiki_pt = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
)

wiki_es = load_dataset(
    "wikimedia/wikipedia",
    "20231101.es",
    split="train",
    streaming=True,
)

dataset_mixed = interleave_datasets(
    [wiki_en, wiki_pt, wiki_es],
    probabilities=[0.50, 0.25, 0.25],
    seed=42,
)


# ------------------------------------------------------------
# 5. Tokenization
# ------------------------------------------------------------

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )


tokenized_dataset = dataset_mixed.map(
    tokenize_function,
    batched=True,
    remove_columns=[
        "id",
        "url",
        "title",
        "text",
    ],
)


# ------------------------------------------------------------
# 6. Collator / DataLoader
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

loader = DataLoader(
    tokenized_dataset,
    batch_size=MICRO_BATCH,
    collate_fn=data_collator,

    # Use zero workers for this diagnostic.
    # This makes stream ordering easier to reason about.
    num_workers=0,
)


# ------------------------------------------------------------
# 7. Frozen evaluation
# ------------------------------------------------------------

micro_losses = []

optimizer_step_losses = []

log_window_losses = []

equivalent_step = 0

print("\n========================================")
print("STARTING FROZEN LOSS TEST")
print("========================================")
print("NO gradients")
print("NO optimizer")
print("NO weight updates")
print()


with torch.inference_mode():

    for micro_step, batch in enumerate(loader, start=1):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            outputs = model(**batch)

        loss = outputs.loss.item()

        micro_losses.append(loss)


        # ----------------------------------------------------
        # Simulate one Trainer optimizer step
        # ----------------------------------------------------

        if len(micro_losses) == GRAD_ACCUM:

            equivalent_step += 1

            step_loss = float(
                np.mean(micro_losses)
            )

            optimizer_step_losses.append(
                step_loss
            )

            log_window_losses.append(
                step_loss
            )

            micro_losses.clear()


            # ------------------------------------------------
            # Equivalent to logging_steps=100
            # ------------------------------------------------

            if equivalent_step % LOGGING_STEPS == 0:

                window_loss = float(
                    np.mean(log_window_losses)
                )

                minimum = float(
                    np.min(log_window_losses)
                )

                maximum = float(
                    np.max(log_window_losses)
                )

                print(
                    f"Equivalent step "
                    f"{equivalent_step:5d} | "
                    f"mean loss = {window_loss:.6f} | "
                    f"min = {minimum:.4f} | "
                    f"max = {maximum:.4f}"
                )

                log_window_losses.clear()


            if equivalent_step >= TEST_STEPS:
                break


print()
print("========================================")
print("FROZEN LOSS TEST COMPLETED")
print("========================================")

GPU: NVIDIA L4

Loading checkpoint:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-106000


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Parameters: 354,823,168

Loading Wikipedia streams...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]


STARTING FROZEN LOSS TEST
NO gradients
NO optimizer
NO weight updates

Equivalent step   100 | mean loss = 2.752875 | min = 2.2633 | max = 2.9478
Equivalent step   200 | mean loss = 2.686952 | min = 2.2218 | max = 2.9339
Equivalent step   300 | mean loss = 2.511525 | min = 2.1626 | max = 2.7397
Equivalent step   400 | mean loss = 2.541561 | min = 2.0858 | max = 2.8022
Equivalent step   500 | mean loss = 2.572391 | min = 2.2613 | max = 2.8445
Equivalent step   600 | mean loss = 2.630158 | min = 2.3931 | max = 2.8278
Equivalent step   700 | mean loss = 2.520425 | min = 2.1049 | max = 2.7846
Equivalent step   800 | mean loss = 2.542685 | min = 2.1420 | max = 2.7979
Equivalent step   900 | mean loss = 2.602480 | min = 2.1763 | max = 2.9043
Equivalent step  1000 | mean loss = 2.557116 | min = 2.1317 | max = 2.8653
Equivalent step  1100 | mean loss = 2.453954 | min = 2.0883 | max = 2.7592
Equivalent step  1200 | mean loss = 2.361959 | min = 1.7154 | max = 2.6501
Equivalent step  1300 | mean

In [ ]:
import os

# Set BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset, interleave_datasets
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)


# ============================================================
# MAIA LITE - CHECKPOINT 106000 RECOVERY EXPERIMENT
#
# PURPOSE:
#   Load ONLY the model weights from checkpoint-106000.
#
#   DO NOT restore:
#     - optimizer
#     - scheduler
#     - Trainer state
#     - gradient scaler
#
#   Keep the original data pipeline and L4 batch configuration.
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"

TOKENIZER_DIR = os.path.join(
    PROJECT_DIR,
    "tokenizador"
)

RECOVERY_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "checkpoints_pretreino",
    "checkpoint-106000"
)

# Completely separate directory.
# checkpoint-106000 will NOT be touched.
RECOVERY_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "recovery_test_106000"
)

RECOVERY_FINAL_DIR = os.path.join(
    RECOVERY_OUTPUT_DIR,
    "final"
)

MAX_LENGTH = 1024

# Number of NEW optimizer steps.
MAX_STEPS = 2000

# Conservative LR for the diagnostic experiment.
LEARNING_RATE = 1e-5

# Fresh optimizer gets a short warmup.
WARMUP_STEPS = 200


# ============================================================
# 1. GPU detection
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. "
        "This recovery experiment should run on GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
device = torch.device("cuda")

props = torch.cuda.get_device_properties(0)
gpu_memory_gb = props.total_memory / (1024 ** 3)

print("=" * 70)
print("MAIA LITE - RECOVERY TRAINING EXPERIMENT")
print("=" * 70)
print(f"GPU:            {gpu_name}")
print(f"GPU memory:     {gpu_memory_gb:.1f} GB")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA:           {torch.version.cuda}")
print()


# ============================================================
# 2. Tokenizer
# ============================================================

print("Loading tokenizer...")

tokenizer = GPT2TokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocabulary: {tokenizer.vocab_size:,}")
print(f"EOS token:            {tokenizer.eos_token_id}")
print(f"PAD token:            {tokenizer.pad_token_id}")
print()


# ============================================================
# 3. Load ONLY model weights from checkpoint-106000
# ============================================================

print("Loading ONLY model weights from:")
print(RECOVERY_CHECKPOINT)
print()

model = GPT2LMHeadModel.from_pretrained(
    RECOVERY_CHECKPOINT,
    local_files_only=True,
)

model.config.use_cache = False

model.to(device)

print(f"Parameters: {model.num_parameters():,}")


# ============================================================
# 4. Verify tied embeddings
# ============================================================

try:

    tied = (
        model.transformer.wte.weight.data_ptr()
        == model.lm_head.weight.data_ptr()
    )

    print(f"Input/output embeddings tied: {tied}")

except Exception as exc:

    print(f"Could not verify tied embeddings: {exc}")


# ============================================================
# 5. Gradient checkpointing
# ============================================================

model.gradient_checkpointing_enable()

print("Gradient checkpointing: ENABLED")


# ============================================================
# 6. Wikipedia streams
#
# IDENTICAL TO ORIGINAL TRAINING.
# ============================================================

print("\nLoading Wikipedia streams...")

wiki_en = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
)

wiki_pt = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
)

wiki_es = load_dataset(
    "wikimedia/wikipedia",
    "20231101.es",
    split="train",
    streaming=True,
)

dataset_mixed = interleave_datasets(
    [wiki_en, wiki_pt, wiki_es],
    probabilities=[0.50, 0.25, 0.25],
    seed=42,
)


# ============================================================
# 7. Tokenization
#
# IDENTICAL TO ORIGINAL TRAINING.
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )


tokenized_dataset = dataset_mixed.map(
    tokenize_function,
    batched=True,
    remove_columns=["id", "url", "title", "text"],
)


# ============================================================
# 8. Causal LM collator
#
# IDENTICAL TO ORIGINAL TRAINING.
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ============================================================
# 9. ORIGINAL L4 batch configuration
# ============================================================

# We deliberately preserve the configuration used during
# the original Maia Lite run.
#
# L4:
#
#   micro batch          = 4
#   gradient accumulation = 8
#   effective batch       = 32

if "L4" in gpu_name:

    batch_size = 4
    gradient_accumulation = 8

    use_bf16 = True
    use_fp16 = False

else:

    # Conservative fallback if Colab unexpectedly assigns
    # another GPU.

    batch_size = 1
    gradient_accumulation = 16

    use_bf16 = False
    use_fp16 = True


effective_batch = batch_size * gradient_accumulation


# ============================================================
# 10. Experiment summary
# ============================================================

print()
print("=" * 70)
print("RECOVERY EXPERIMENT CONFIGURATION")
print("=" * 70)

print(f"Checkpoint weights:      checkpoint-106000")
print()
print("RESTORED:")
print("  Model weights:         YES")
print()
print("NOT RESTORED:")
print("  Optimizer:             NEW")
print("  LR scheduler:          NEW")
print("  Trainer state:         NEW")
print("  Gradient scaler:       NEW")
print()

print("DATA:")
print("  Wikipedia EN:          50%")
print("  Wikipedia PT:          25%")
print("  Wikipedia ES:          25%")
print(f"  Context length:        {MAX_LENGTH}")
print()

print("BATCH:")
print(f"  Micro batch:           {batch_size}")
print(f"  Gradient accumulation: {gradient_accumulation}")
print(f"  Effective batch:       {effective_batch}")
print()

print("OPTIMIZATION:")
print(f"  Learning rate:         {LEARNING_RATE:.2e}")
print(f"  Warmup steps:          {WARMUP_STEPS}")
print("  Scheduler:             linear")
print("  Weight decay:          0.01")
print()

print(f"Recovery steps:          {MAX_STEPS}")
print(f"BF16:                    {use_bf16}")
print(f"FP16:                    {use_fp16}")

print("=" * 70)


# ============================================================
# 11. Training arguments
# ============================================================

training_args = TrainingArguments(

    # IMPORTANT:
    # Separate from the production checkpoint directory.
    output_dir=RECOVERY_OUTPUT_DIR,

    # These are NEW steps, starting logically from zero.
    max_steps=MAX_STEPS,

    # Preserve original L4 batching.
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation,

    # Precision.
    bf16=use_bf16,
    fp16=use_fp16,

    # --------------------------------------------------------
    # NEW OPTIMIZATION STATE
    # --------------------------------------------------------

    learning_rate=LEARNING_RATE,

    weight_decay=0.01,

    lr_scheduler_type="linear",

    warmup_steps=WARMUP_STEPS,

    # Gradient clipping provides another safety barrier.
    max_grad_norm=1.0,

    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=100,

    logging_first_step=True,

    # --------------------------------------------------------
    # CHECKPOINTS
    # --------------------------------------------------------

    # Save frequently because this is a diagnostic experiment.
    save_strategy="steps",

    save_steps=500,

    # Keep all four diagnostic checkpoints.
    save_total_limit=4,

    # --------------------------------------------------------
    # STREAMING DATASET
    # --------------------------------------------------------

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    remove_unused_columns=True,

    # There is no previous Trainer state to skip.
    # Still harmless and consistent with streaming behavior.
    ignore_data_skip=True,

    # --------------------------------------------------------
    # OTHER
    # --------------------------------------------------------

    report_to="none",

    seed=42,
)


# ============================================================
# 12. Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


# ============================================================
# 13. CUDA diagnostics
# ============================================================

torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / (1024 ** 3)
reserved = torch.cuda.memory_reserved() / (1024 ** 3)

print()
print("CUDA memory before training")
print("---------------------------")
print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved:  {reserved:.2f} GB")


# ============================================================
# 14. START RECOVERY TRAINING
#
# CRITICAL:
#
# DO NOT USE:
#
# trainer.train(
#     resume_from_checkpoint=RECOVERY_CHECKPOINT
# )
#
# We intentionally call train() WITHOUT a checkpoint.
#
# This forces Trainer to create:
#
#   NEW AdamW optimizer
#   NEW LR scheduler
#   NEW training state
#
# while keeping the loaded checkpoint-106000 weights.
# ============================================================

print()
print("=" * 70)
print("STARTING RECOVERY TRAINING")
print("=" * 70)

print()
print("Checkpoint-106000 provides ONLY the model weights.")
print()

print("Watch the following columns:")
print()
print("  Step")
print("  Training Loss")
print("  Learning Rate")
print()

print("The experiment is successful if loss remains stable")
print("around the checkpoint's healthy region.")
print()

print("WARNING:")
print("If loss shows sustained monotonic growth toward 3.0,")
print("stop the experiment.")
print()

print("=" * 70)
print()


train_result = trainer.train()


# ============================================================
# 15. Save final recovery model
# ============================================================

trainer.save_model(RECOVERY_FINAL_DIR)

tokenizer.save_pretrained(
    RECOVERY_FINAL_DIR
)


# ============================================================
# 16. Print log history
# ============================================================

print()
print("=" * 70)
print("RECOVERY TRAINING LOG")
print("=" * 70)

for entry in trainer.state.log_history:

    if "loss" in entry:

        step = entry.get("step", "?")
        loss = entry.get("loss", None)
        lr = entry.get("learning_rate", None)

        if loss is not None:

            if lr is not None:

                print(
                    f"Step {step:5} | "
                    f"loss = {loss:.6f} | "
                    f"LR = {lr:.8e}"
                )

            else:

                print(
                    f"Step {step:5} | "
                    f"loss = {loss:.6f}"
                )


print()
print("=" * 70)
print("RECOVERY EXPERIMENT COMPLETED")
print("=" * 70)
print(f"Final recovery model:")
print(RECOVERY_FINAL_DIR)
print()
print("Original checkpoint-106000 was NOT modified.")
print("=" * 70)

MAIA LITE - RECOVERY TRAINING EXPERIMENT
GPU:            NVIDIA L4
GPU memory:     22.0 GB
PyTorch:        2.11.0+cu128
CUDA:           12.8

Loading tokenizer...
Tokenizer vocabulary: 50,257
EOS token:            2
PAD token:            1

Loading ONLY model weights from:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-106000



Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Parameters: 354,823,168
Input/output embeddings tied: True
Gradient checkpointing: ENABLED

Loading Wikipedia streams...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]


RECOVERY EXPERIMENT CONFIGURATION
Checkpoint weights:      checkpoint-106000

RESTORED:
  Model weights:         YES

NOT RESTORED:
  Optimizer:             NEW
  LR scheduler:          NEW
  Trainer state:         NEW
  Gradient scaler:       NEW

DATA:
  Wikipedia EN:          50%
  Wikipedia PT:          25%
  Wikipedia ES:          25%
  Context length:        1024

BATCH:
  Micro batch:           4
  Gradient accumulation: 8
  Effective batch:       32

OPTIMIZATION:
  Learning rate:         1.00e-05
  Warmup steps:          200
  Scheduler:             linear
  Weight decay:          0.01

Recovery steps:          2000
BF16:                    True
FP16:                    False

CUDA memory before training
---------------------------
Allocated: 4.25 GB
Reserved:  4.42 GB

STARTING RECOVERY TRAINING

Checkpoint-106000 provides ONLY the model weights.

Watch the following columns:

  Step
  Training Loss
  Learning Rate

The experiment is successful if loss remains stable
around 

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RECOVERY TRAINING LOG
Step     1 | loss = 2.799536 | LR = 0.00000000e+00
Step   100 | loss = 2.745501 | LR = 4.95000000e-06
Step   200 | loss = 2.785903 | LR = 9.95000000e-06
Step   300 | loss = 2.757358 | LR = 9.45000000e-06
Step   400 | loss = 2.635333 | LR = 8.89444444e-06
Step   500 | loss = 2.558901 | LR = 8.33888889e-06
Step   600 | loss = 2.585332 | LR = 7.78333333e-06
Step   700 | loss = 2.613076 | LR = 7.22777778e-06
Step   800 | loss = 2.557166 | LR = 6.67222222e-06
Step   900 | loss = 2.628467 | LR = 6.11666667e-06
Step  1000 | loss = 2.667072 | LR = 5.56111111e-06
Step  1100 | loss = 2.703328 | LR = 5.00555556e-06
Step  1200 | loss = 2.686302 | LR = 4.45000000e-06
Step  1300 | loss = 2.656910 | LR = 3.89444444e-06
Step  1400 | loss = 2.622562 | LR = 3.33888889e-06
Step  1500 | loss = 2.649464 | LR = 2.78333333e-06
Step  1600 | loss = 2.632236 | LR = 2.22777778e-06
Step  1700 | loss = 2.659712 | LR = 1.67222222e-06
Step  1800 | loss = 2.690224 | LR = 1.11666667e-06
Step  19

In [ ]:
import os

# ============================================================
# MAIA LITE - DEFINITIVE A100 PRETRAINING CONTINUATION
# ============================================================
#
# FIRST EXECUTION:
#   - Load ONLY model weights from checkpoint-106000
#   - NEW optimizer
#   - NEW scheduler
#   - NEW Trainer state
#
# SUBSEQUENT COLAB SESSIONS:
#   - Resume latest checkpoint created by THIS continuation
#   - Restore model + optimizer + scheduler + Trainer state
#
# DATA PIPELINE:
#   IDENTICAL to the successful recovery experiment.
#
# ============================================================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math
import torch

from datasets import load_dataset, interleave_datasets

from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
)

from transformers.trainer_utils import get_last_checkpoint


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"

TOKENIZER_DIR = os.path.join(
    PROJECT_DIR,
    "tokenizador"
)

# GOLDEN checkpoint.
# This directory is NEVER used as output_dir.
GOLDEN_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "checkpoints_pretreino",
    "checkpoint-106000"
)

# New independent continuation.
CONTINUATION_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints_continuacao_a100"
)

CURRENT_MODEL_DIR = os.path.join(
    CONTINUATION_DIR,
    "current_model"
)

os.makedirs(
    CONTINUATION_DIR,
    exist_ok=True
)


# ============================================================
# 2. LONG-TERM TRAINING PLAN
# ============================================================

ORIGINAL_STEP = 106_000

TARGET_ORIGINAL_STEP = 300_000

CONTINUATION_STEPS = (
    TARGET_ORIGINAL_STEP
    - ORIGINAL_STEP
)

assert CONTINUATION_STEPS == 194_000


# ============================================================
# 3. DATA
# ============================================================

MAX_LENGTH = 1024

SEED = 42


# ============================================================
# 4. A100 BATCH CONFIGURATION
# ============================================================
#
# Successful L4 recovery:
#
#   micro batch           = 4
#   gradient accumulation = 8
#   effective batch       = 32
#
# A100:
#
#   micro batch           = 8
#   gradient accumulation = 4
#   effective batch       = 32
#
# Therefore the effective batch remains unchanged.
# ============================================================

BATCH_SIZE = 8

GRADIENT_ACCUMULATION = 4

EFFECTIVE_BATCH = (
    BATCH_SIZE
    * GRADIENT_ACCUMULATION
)

assert EFFECTIVE_BATCH == 32


# ============================================================
# 5. OPTIMIZATION
# ============================================================

LEARNING_RATE = 1.0e-5

WARMUP_STEPS = 500

WEIGHT_DECAY = 0.01

MAX_GRAD_NORM = 1.0


# ============================================================
# 6. LOGGING / CHECKPOINTS
# ============================================================

LOGGING_STEPS = 100

SAVE_STEPS = 1000

SAVE_TOTAL_LIMIT = 3


# ============================================================
# 7. GPU DETECTION
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU not available."
    )


gpu_name = torch.cuda.get_device_name(0)

device = torch.device("cuda")

props = torch.cuda.get_device_properties(0)

gpu_memory_gb = (
    props.total_memory
    / (1024 ** 3)
)


print("=" * 74)

print(
    "MAIA LITE - DEFINITIVE A100 "
    "PRETRAINING CONTINUATION"
)

print("=" * 74)

print(f"GPU:                 {gpu_name}")
print(f"GPU memory:          {gpu_memory_gb:.1f} GB")
print(f"PyTorch:             {torch.__version__}")
print(f"CUDA:                {torch.version.cuda}")

print()


if "A100" not in gpu_name.upper():

    print("WARNING:")
    print(
        "This configuration was designed "
        "for NVIDIA A100."
    )

    print(
        f"Detected GPU: {gpu_name}"
    )

    print()


# ============================================================
# 8. TOKENIZER
# ============================================================

print("Loading tokenizer...")


tokenizer = GPT2TokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True,
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token


print(
    f"Tokenizer vocabulary: "
    f"{tokenizer.vocab_size:,}"
)

print(
    f"EOS token:            "
    f"{tokenizer.eos_token_id}"
)

print(
    f"PAD token:            "
    f"{tokenizer.pad_token_id}"
)

print()


# ============================================================
# 9. DETECT EXISTING CONTINUATION
# ============================================================

resume_checkpoint = get_last_checkpoint(
    CONTINUATION_DIR
)


# ============================================================
# 10. LOAD MODEL
# ============================================================

if resume_checkpoint is None:

    # --------------------------------------------------------
    # FIRST EXECUTION
    # --------------------------------------------------------

    print("=" * 74)
    print("NEW A100 CONTINUATION")
    print("=" * 74)

    print()

    print(
        "Loading ONLY model weights from:"
    )

    print(GOLDEN_CHECKPOINT)

    print()


    model = GPT2LMHeadModel.from_pretrained(
        GOLDEN_CHECKPOINT,
        local_files_only=True,
    )


    print("RESTORED:")
    print("  Model weights:         YES")

    print()

    print("NOT RESTORED:")
    print("  Optimizer:             NEW")
    print("  LR scheduler:          NEW")
    print("  Trainer state:         NEW")
    print("  Gradient scaler:       NEW")


else:

    # --------------------------------------------------------
    # SUBSEQUENT COLAB SESSION
    # --------------------------------------------------------

    print("=" * 74)
    print("RESUMING A100 CONTINUATION")
    print("=" * 74)

    print()

    print(
        "Latest continuation checkpoint:"
    )

    print(resume_checkpoint)

    print()


    model = GPT2LMHeadModel.from_pretrained(
        resume_checkpoint,
        local_files_only=True,
    )


    print("RESTORING:")
    print("  Model weights")
    print("  Optimizer")
    print("  LR scheduler")
    print("  Trainer state")


# ============================================================
# 11. MODEL CONFIGURATION
# ============================================================

model.config.use_cache = False

model.to(device)

model.gradient_checkpointing_enable()


print()

print(
    f"Parameters:            "
    f"{model.num_parameters():,}"
)


# Verify tied embeddings.

try:

    tied = (
        model.transformer.wte.weight.data_ptr()
        ==
        model.lm_head.weight.data_ptr()
    )

    print(
        f"Input/output embeddings tied: "
        f"{tied}"
    )

except Exception as exc:

    print(
        "Could not verify tied embeddings:",
        exc
    )


print(
    "Gradient checkpointing: ENABLED"
)

print()


# ============================================================
# 12. WIKIPEDIA STREAMS
#
# EXACTLY THE SAME PIPELINE AS THE SUCCESSFUL RECOVERY TEST.
# ============================================================

print(
    "Loading Wikipedia streams..."
)


wiki_en = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
)


wiki_pt = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
)


wiki_es = load_dataset(
    "wikimedia/wikipedia",
    "20231101.es",
    split="train",
    streaming=True,
)


dataset_mixed = interleave_datasets(

    [
        wiki_en,
        wiki_pt,
        wiki_es
    ],

    probabilities=[
        0.50,
        0.25,
        0.25
    ],

    seed=SEED,
)


# ============================================================
# 13. TOKENIZATION
#
# EXACTLY THE SAME AS THE SUCCESSFUL RECOVERY TEST.
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )


tokenized_dataset = dataset_mixed.map(

    tokenize_function,

    batched=True,

    remove_columns=[
        "id",
        "url",
        "title",
        "text"
    ],
)


# ============================================================
# 14. DATA COLLATOR
#
# IDENTICAL TO SUCCESSFUL RECOVERY.
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ============================================================
# 15. TRAINING MONITOR
# ============================================================

class MaiaTrainingMonitor(
    TrainerCallback
):

    def __init__(self):

        self.recent_losses = []


    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):

        if not logs:
            return

        loss = logs.get("loss")

        lr = logs.get(
            "learning_rate"
        )

        grad_norm = logs.get(
            "grad_norm"
        )


        if loss is None:
            return


        continuation_step = (
            state.global_step
        )

        equivalent_step = (
            ORIGINAL_STEP
            + continuation_step
        )


        self.recent_losses.append(
            float(loss)
        )


        if len(
            self.recent_losses
        ) > 10:

            self.recent_losses.pop(0)


        message = (

            f"[MAIA] "
            f"continuation="
            f"{continuation_step:6d} | "

            f"equivalent="
            f"{equivalent_step:6d} | "

            f"loss="
            f"{loss:.6f}"
        )


        if lr is not None:

            message += (
                f" | LR="
                f"{lr:.3e}"
            )


        if grad_norm is not None:

            message += (
                f" | grad_norm="
                f"{grad_norm:.4f}"
            )


        print()
        print(message)


        # ----------------------------------------------------
        # NON-FINITE LOSS
        # ----------------------------------------------------

        if not math.isfinite(
            float(loss)
        ):

            print()
            print("!" * 60)

            print(
                "NON-FINITE LOSS DETECTED."
            )

            print(
                "TRAINING WILL STOP."
            )

            print("!" * 60)


            control.should_training_stop = True

            return


        # ----------------------------------------------------
        # SUSTAINED HIGH LOSS PROTECTION
        #
        # Stop only if FIVE consecutive 100-step averages
        # exceed 3.20.
        # ----------------------------------------------------

        if len(
            self.recent_losses
        ) >= 5:

            last5 = (
                self.recent_losses[-5:]
            )


            if all(
                x > 3.20
                for x in last5
            ):

                print()
                print("!" * 60)

                print(
                    "SUSTAINED HIGH LOSS DETECTED."
                )

                print(
                    "Last five logged losses:",
                    [
                        round(x, 4)
                        for x in last5
                    ]
                )

                print(
                    "TRAINING WILL STOP."
                )

                print("!" * 60)


                control.should_training_stop = True


# ============================================================
# 16. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    output_dir=CONTINUATION_DIR,


    # --------------------------------------------------------
    # LONG-TERM TRAINING
    # --------------------------------------------------------

    max_steps=CONTINUATION_STEPS,


    # --------------------------------------------------------
    # A100 BATCH
    # --------------------------------------------------------

    per_device_train_batch_size=(
        BATCH_SIZE
    ),

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION
    ),


    # --------------------------------------------------------
    # PRECISION
    # --------------------------------------------------------

    bf16=True,

    fp16=False,


    # --------------------------------------------------------
    # OPTIMIZATION
    # --------------------------------------------------------

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    lr_scheduler_type="cosine",

    warmup_steps=WARMUP_STEPS,

    max_grad_norm=MAX_GRAD_NORM,

    optim="adamw_torch",


    # --------------------------------------------------------
    # MEMORY
    # --------------------------------------------------------

    gradient_checkpointing=True,


    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=LOGGING_STEPS,

    logging_first_step=True,


    # --------------------------------------------------------
    # CHECKPOINTS
    # --------------------------------------------------------

    save_strategy="steps",

    save_steps=SAVE_STEPS,

    save_total_limit=SAVE_TOTAL_LIMIT,


    # --------------------------------------------------------
    # STREAMING DATASET
    #
    # SAME AS SUCCESSFUL RECOVERY.
    # --------------------------------------------------------

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    remove_unused_columns=True,

    ignore_data_skip=True,


    # --------------------------------------------------------
    # OTHER
    # --------------------------------------------------------

    report_to="none",

    seed=SEED,
)


# ============================================================
# 17. TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset,

    data_collator=data_collator,

    callbacks=[
        MaiaTrainingMonitor()
    ],
)


# ============================================================
# 18. CONFIGURATION SUMMARY
# ============================================================

print()
print("=" * 74)

print(
    "MAIA LITE - A100 TRAINING CONFIGURATION"
)

print("=" * 74)

print()


print("TRAINING PLAN:")

print(
    f"  Golden checkpoint:      "
    f"{ORIGINAL_STEP:,}"
)

print(
    f"  Target equivalent:      "
    f"{TARGET_ORIGINAL_STEP:,}"
)

print(
    f"  New optimizer steps:    "
    f"{CONTINUATION_STEPS:,}"
)

print()


print("DATA:")

print(
    "  Wikipedia EN:           50%"
)

print(
    "  Wikipedia PT:           25%"
)

print(
    "  Wikipedia ES:           25%"
)

print(
    f"  Context length:         "
    f"{MAX_LENGTH}"
)

print()


print("A100 BATCH:")

print(
    f"  Micro batch:            "
    f"{BATCH_SIZE}"
)

print(
    f"  Gradient accumulation:  "
    f"{GRADIENT_ACCUMULATION}"
)

print(
    f"  Effective batch:        "
    f"{EFFECTIVE_BATCH}"
)

print()


print("OPTIMIZATION:")

print(
    f"  Peak learning rate:     "
    f"{LEARNING_RATE:.2e}"
)

print(
    f"  Warmup steps:           "
    f"{WARMUP_STEPS}"
)

print(
    "  Scheduler:              cosine"
)

print(
    f"  Weight decay:           "
    f"{WEIGHT_DECAY}"
)

print(
    f"  Max gradient norm:      "
    f"{MAX_GRAD_NORM}"
)

print()


print("CHECKPOINTS:")

print(
    f"  Directory:              "
    f"{CONTINUATION_DIR}"
)

print(
    f"  Save every:             "
    f"{SAVE_STEPS:,} steps"
)

print(
    f"  Keep latest:            "
    f"{SAVE_TOTAL_LIMIT}"
)

print()


if resume_checkpoint is None:

    print("MODE:")
    print(
        "  GOLDEN checkpoint weights ONLY"
    )

    print(
        "  NEW optimizer"
    )

    print(
        "  NEW scheduler"
    )

    print(
        "  NEW Trainer state"
    )

else:

    print("MODE:")

    print(
        "  RESUME HEALTHY CONTINUATION"
    )

    print(
        f"  {resume_checkpoint}"
    )


print()
print("=" * 74)


# ============================================================
# 19. CUDA MEMORY
# ============================================================

torch.cuda.empty_cache()


allocated = (
    torch.cuda.memory_allocated()
    / (1024 ** 3)
)

reserved = (
    torch.cuda.memory_reserved()
    / (1024 ** 3)
)


print()

print(
    "CUDA memory before training"
)

print(
    "---------------------------"
)

print(
    f"Allocated: {allocated:.2f} GB"
)

print(
    f"Reserved:  {reserved:.2f} GB"
)

print()


# ============================================================
# 20. START TRAINING
# ============================================================

print("=" * 74)

print(
    "STARTING MAIA LITE A100 TRAINING"
)

print("=" * 74)

print()


if resume_checkpoint is None:

    print(
        "Starting from checkpoint-106000 "
        "MODEL WEIGHTS ONLY."
    )

    print()

    print(
        "Optimizer and scheduler are NEW."
    )

else:

    print(
        "Restoring complete healthy "
        "continuation state from:"
    )

    print()

    print(
        resume_checkpoint
    )


print()

print(
    "Monitor:"
)

print(
    "  loss"
)

print(
    "  learning rate"
)

print(
    "  gradient norm"
)

print()

print(
    "Golden checkpoint-106000 "
    "will NOT be modified."
)

print()

print("=" * 74)

print()


# ============================================================
# 21. TRAIN
# ============================================================

if resume_checkpoint is None:

    # --------------------------------------------------------
    # FIRST RUN
    #
    # CRITICAL:
    #
    # Do NOT use resume_from_checkpoint here.
    #
    # This creates a NEW optimizer and scheduler.
    # --------------------------------------------------------

    train_result = trainer.train()


else:

    # --------------------------------------------------------
    # FUTURE COLAB SESSIONS
    #
    # Restore optimizer/scheduler/state generated by this
    # healthy continuation.
    # --------------------------------------------------------

    train_result = trainer.train(
        resume_from_checkpoint=(
            resume_checkpoint
        )
    )


# ============================================================
# 22. SAVE CURRENT MODEL
# ============================================================

trainer.save_model(
    CURRENT_MODEL_DIR
)

tokenizer.save_pretrained(
    CURRENT_MODEL_DIR
)


# ============================================================
# 23. SESSION SUMMARY
# ============================================================

continuation_step = (
    trainer.state.global_step
)

equivalent_step = (
    ORIGINAL_STEP
    + continuation_step
)


print()
print("=" * 74)

print(
    "MAIA LITE A100 SESSION COMPLETED"
)

print("=" * 74)

print()


print(
    f"Continuation step:        "
    f"{continuation_step:,}"
)

print(
    f"Equivalent original step: "
    f"{equivalent_step:,}"
)

print()


print(
    "Current model:"
)

print(
    CURRENT_MODEL_DIR
)

print()


print(
    "Golden checkpoint-106000 "
    "remains untouched."
)

print("=" * 74)

MAIA LITE - DEFINITIVE A100 PRETRAINING CONTINUATION
GPU:                 NVIDIA A100-SXM4-80GB
GPU memory:          79.3 GB
PyTorch:             2.11.0+cu128
CUDA:                12.8

Loading tokenizer...
Tokenizer vocabulary: 50,257
EOS token:            2
PAD token:            1

NEW A100 CONTINUATION

Loading ONLY model weights from:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-106000



Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

RESTORED:
  Model weights:         YES

NOT RESTORED:
  Optimizer:             NEW
  LR scheduler:          NEW
  Trainer state:         NEW
  Gradient scaler:       NEW

Parameters:            354,823,168
Input/output embeddings tied: True
Gradient checkpointing: ENABLED

Loading Wikipedia streams...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

...


[MAIA] continuation=  7100 | equivalent=113100 | loss=2.569692 | LR=9.971e-06 | grad_norm=0.3284

[MAIA] continuation=  7200 | equivalent=113200 | loss=2.561740 | LR=9.970e-06 | grad_norm=0.2908

[MAIA] continuation=  7300 | equivalent=113300 | loss=2.555475 | LR=9.970e-06 | grad_norm=0.3314

[MAIA] continuation=  7400 | equivalent=113400 | loss=2.575822 | LR=9.969e-06 | grad_norm=0.2985

[MAIA] continuation=  7500 | equivalent=113500 | loss=2.561971 | LR=9.968e-06 | grad_norm=0.2980

[MAIA] continuation=  7600 | equivalent=113600 | loss=2.571344 | LR=9.967e-06 | grad_norm=0.3123

[MAIA] continuation=  7700 | equivalent=113700 | loss=2.576125 | LR=9.966e-06 | grad_norm=0.2812

[MAIA] continuation=  7800 | equivalent=113800 | loss=2.536530 | LR=9.965e-06 | grad_norm=0.3391

[MAIA] continuation=  7900 | equivalent=113900 | loss=2.541058 | LR=9.964e-06 | grad_norm=0.3261

[MAIA] continuation=  8000 | equivalent=114000 | loss=2.471693 | LR=9.963e-06 | grad_norm=0.2954


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[MAIA] continuation=  8100 | equivalent=114100 | loss=2.420596 | LR=9.962e-06 | grad_norm=0.3007

[MAIA] continuation=  8200 | equivalent=114200 | loss=2.413096 | LR=9.961e-06 | grad_norm=0.3261

[MAIA] continuation=  8300 | equivalent=114300 | loss=2.455498 | LR=9.960e-06 | grad_norm=0.2843

[MAIA] continuation=  8400 | equivalent=114400 | loss=2.483626 | LR=9.959e-06 | grad_norm=0.4557

[MAIA] continuation=  8500 | equivalent=114500 | loss=2.435156 | LR=9.958e-06 | grad_norm=0.3698

[MAIA] continuation=  8600 | equivalent=114600 | loss=2.401397 | LR=9.957e-06 | grad_norm=0.3065

[MAIA] continuation=  8700 | equivalent=114700 | loss=2.481856 | LR=9.956e-06 | grad_norm=0.2979

[MAIA] continuation=  8800 | equivalent=114800 | loss=2.470807 | LR=9.955e-06 | grad_norm=0.3693

[MAIA] continuation=  8900 | equivalent=114900 | loss=2.456309 | LR=9.954e-06 | grad_norm=0.3107

[MAIA] continuation=  9000 | equivalent=115000 | loss=2.489367 | LR=9.952e-06 | grad_norm=0.3254


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
1,2.792798
100,2.747724
200,2.792141
300,2.766445
400,2.644399
500,2.564493
600,2.590161
700,2.616452
800,2.560267
900,2.631811
